# Data exploring

In [ ]:
import pandas as pd

In [ ]:
# TODO: Understand connection between all 3 files

## Check bookings csv

In [ ]:
bookings = pd.read_csv('../data/20260108_bookings.csv')

In [ ]:
print(bookings.shape)
bookings.head()

In [ ]:
bookings[(bookings['product_id'] == 'XRJA3') & (bookings['date'] == 20260213)]

In [ ]:
bookings['id'].value_counts()[lambda x: x > 1]
# id columns is not unique?!?!?

In [ ]:
bookings[bookings['id'] == 733716890926]

In [ ]:
bookings[bookings['id'] == 679335503950]

- Id column does not make sense to me!

In [ ]:
# Check for duplicates
print(len(bookings))
print(len(bookings[bookings.duplicated(keep=False)]))
print(len(bookings[bookings.duplicated(keep='first')]))

len(bookings.drop_duplicates(keep='first'))

In [ ]:
# Check ids
print(len(bookings['id'].unique()))
print(len(bookings['bkg_id'].unique()))
print(len(bookings['cust_id'].unique()))

In [ ]:
# Investigate dates

print(bookings['date'].dtype)
# Formatted as int -> Format to datetime


print(bookings['date'].min())
print(bookings['date'].max(), '\n')
# -> Bookings from 2023/04/21 to 2027/04/19

print(bookings['booking_creation_date'].min())
print(bookings['booking_creation_date'].max())
# -> Bookings created from 2018/10/02 to 2025/12/22

In [ ]:
# Cancelled Bookings should be ignored when ingesting data
print(bookings['status'].unique())

print(len(bookings[bookings['status'] == 'Cancelled']) / len(bookings['status']) * 100)
print(len(bookings[bookings['status'] == 'Cancelled']))

In [ ]:
# Customer id should be ignored when ingesting data
print(f'Unique customers: {len(bookings["cust_id"].unique())}')

# Connection to booking ids?!
print(f'Unique bookings: {len(bookings["bkg_id"].unique())}')

In [ ]:
# Check features
print(f'Feature 1: \n{bookings["feature_1"].unique()} \n')
print(
    f'Feature 2: \n{bookings["feature_2"].unique()}'
)  # Can be ingnored in data ingestion

In [ ]:
print(
    f'Currencies: {bookings["base_currency"].unique()} \n'
)  # Pfund Sterling (Brit. Pounds) and Euros

print(bookings['gross_revenue'].min())  # Negative values possible
print(bookings['gross_revenue'].max())
print(bookings['gross_revenue'].mean(), '\n')

print(bookings['net_revenue'].min())  # Negative values possible
print(bookings['net_revenue'].max())
print(bookings['net_revenue'].mean())

# bookings[bookings['net_revenue'] == -709.4] # High discount
bookings[bookings['gross_revenue'] == -84.0]  # Looks like a gift :)

## Check capacity csv

In [ ]:
capacity = pd.read_csv('../data/20260108_capacity.csv')

In [ ]:
print(capacity.shape)
capacity.head()
# One line per "bookable unit"

In [ ]:
capacity[(capacity['product_id'] == 'XRJA3')].sort_values(
    by='calendar_date'
)  # & (capacity['calendar_date'] == '20271213')

In [ ]:
len(capacity['id'].unique())
# Here, the id looks fine

In [ ]:
capacity[capacity['id'] == '65851f3235']

In [ ]:
print(len(capacity['product_id'].unique()))
# 780 different apartment types

print(len(capacity['calendar_date'].unique()))
print(capacity['calendar_date'].min())
print(capacity['calendar_date'].max())

print(len(capacity['product_id'].unique()) * len(capacity['calendar_date'].unique()))
# 780 * 1702 = 1327560
# Every product_id has every calendar date listed -> one line per bookable unit

In [ ]:
print(f'Is bookable unique values: {capacity["is_bookable"].unique()}\n')
# Count of how many bookable units (product_id + date) are available

print(f'Is option unique values: {capacity["is_option"].unique()}')
# Count of how many bookable units are reservated as optional

# Negativ values = Overbookings?!
# Sum of is_bookable and is_option equals the available units for new customers

## Check prices csv

- ID represents the product_id and date (bookable unit) -> 110.336 unique units
- 264 unique product_ids -> Less then in capacity
- Test hypothese: Every product_id as 552 rows with the same date+length_of_stay pattern
- 489 unique dates -> Less then in capacity

In [ ]:
prices = pd.read_csv('../data/20260108_prices.csv')

In [ ]:
print(prices.shape)
prices.head()

In [ ]:
print(len(prices['id'].unique()))
print(f'Len unique product ids: {len(prices["product_id"].unique())}')
print(f'Len unique dates: {len(prices["date"].unique())}')

In [ ]:
prices[prices['id'] == '65851f3235'].head()
# ID represents the product_id and date (bookable unit)

In [ ]:
df = prices[(prices['product_id'] == 'IXQD32')].sort_values(by='date')
df
# date + length_of_stay = date of next row
# TODO: Evaluate pattern of product_ids - I see one

In [ ]:
# Dates
print(f'First date available: {prices["date"].min()}')
print(f'Last data available: {prices["date"].max()}\n')

# Prices
print(f'Min price available: {prices["current_price"].min()}')
print(f'Max price available: {prices["current_price"].max()}\n')

# length_of_stay
print(prices['length_of_stay'].unique())  # XN equals to Extra Night?